In [1]:
from ibis import _
import pandas as pd

import src
from src.load import DataLoader

In [2]:
pd.set_option("display.max_rows", 1024)
pd.set_option("display.max_colwidth", 256)

In [3]:
dl = DataLoader()

In [11]:
videos = (
    dl.videos(filtered=True)
    .join(dl.channels().filter(_.channel != "FDP"), "channel_id")
    .select(
        ["video_id", "channel", "video_likes", "video_views", "video_uploadtime", "video_title"],
    )
    .to_pandas()
)

sents = dl.sentences(filtered=True).join(dl.popbert(filtered=True), "sentence_id").to_pandas()

sents = sents.groupby("video_id", observed=True).agg(
    n_sentences=("video_id", "size"),
    elite=("elite", "mean"),
    pplcentr=("pplcentr", "mean"),
)

videos = videos.merge(sents, on="video_id")

# Most Liked

In [ ]:
quantiles = videos.groupby("channel").video_likes.quantile(q=0.99).rename("quantile")

In [ ]:
df = videos.merge(quantiles, how="left", on="channel")
df["top_1p"] = df.apply(lambda x: 1 if x.video_likes > x["quantile"] else 0, axis=1)

In [ ]:
top_videos = (
    df.sort_values(["channel", "video_likes"], ascending=False)
    .groupby("channel")
    .head(
        10,
    )
    .set_index(["channel", "video_id"])
)

In [ ]:
top_videos

In [ ]:
top_videos.reset_index()[["channel", "video_likes", "video_views", "video_title"]].to_csv(
    src.OUT / "tables/top_videos_per_channel.csv",
    index=False,
)

# Most Anti-Elitism

In [12]:
quantiles = videos.groupby("channel").elite.quantile(q=0.99).rename("quantile")

In [13]:
df = videos.merge(quantiles, how="left", on="channel")
df["top_1p"] = df.apply(lambda x: 1 if x.elite > x["quantile"] else 0, axis=1)

In [14]:
top_videos = (
    df.sort_values(["channel", "elite"], ascending=False)
    .groupby("channel")
    .head(
        10,
    )
    .set_index(["channel", "video_id"])
)

In [15]:
top_videos

video_likes  video_views video_uploadtime  \
channel video_id                                                 
SPD     F0c4_NPpnGA          142         2480       2024-01-18   
        d6vy8DvlA6k          177         2811       2020-03-05   
        2eJPpGsTI0c          180         2606       2021-08-30   
        2rwXzpbyDEU          165         3675       2020-03-05   
        y1WnueSc9fQ           61          839       2021-05-18   
        3HwEQBSzMUI           32          967       2023-12-09   
        B4hGX4jIMks          198         4516       2024-01-12   
        CtIAZCMJyWg          144         5122       2022-03-25   
        nuVHZu54xvU          111         2109       2021-09-18   
        67wvrzYddRo           77          493       2021-08-14   
Left    l5abgjJlcE4         1692        37786       2023-05-09   
        MUgDgKGu5Bg          431         7201       2021-09-02   
        9DC619q6mVA           24          630       2019-04-05   
        FFg9z4i9lqM          119         2704       2022-12-06   
        35yCb1A9bmc         6428       230089       2019-11-07   
        GSfgp8r9h3A           91         2762       2018-06-18   
        s15ADZa3BvI          177         2270       2023-09-21   
        L3wllg-iZNg          109         1488       2022-11-25   
        zN5YQIN5ep8          168         3614       2021-09-20   
        quIkOTiaCSA           75          989       2018-09-26   
Greens  BKA5Rs9r7WE            5          281       2018-01-27   
        iM80OK5_e9E           13          498       2018-01-27   
        sTFRFy4jcN4           22         1868       2018-11-10   
        fvgOxVAw5hQ            6          212       2018-01-27   
        kMlQJQjV__M          194         6922       2019-09-20   
        5aAfx-1usQ0           84         1113       2018-06-29   
        neJCKmYZZBc           42         1133       2019-11-16   
        QO_Y14cx8Lo           51          774       2021-06-13   
        _cfV76TxsY8           15         1722       2019-11-16   
        YjN_Q4JIQjw           41         2924       2022-10-14   
CSU     toVAryo-KIA          216         4520       2019-09-30   
        nDYYxXPqXGQ           71         5213       2023-02-22   
        N431nIgfOnA           21          641       2020-02-05   
        K0HX5OUY1cQ          172         3809       2019-11-20   
        iPKZDFAdIYs           28          472       2019-10-05   
        983lQr9XElg           82         2932       2021-09-16   
        16jEAhKmF-I           35          955       2021-09-25   
        6eSiwakBcgY           21          323       2021-09-20   
        pJVOIfs3h-k           29         2884       2019-03-31   
        -YMkFuGjPr4           99         4187       2021-08-31   
CDU     EksM0bVVH1M           36         1763       2020-02-05   
        bbcjzNNtveo           18          334       2022-01-19   
        xgkXD9s4Opc          543        20013       2020-02-26   
        oYAl63O10gU           15          693       2019-04-08   
        6_RMW7sF4kM           69         2893       2023-06-20   
        t9oeOo08rGM            8          190       2022-01-24   
        q64peGPwHV8           46          965       2021-09-22   
        GfIbnaLOS7M          174         4009       2021-09-08   
        mINqf00pBts           18          569       2019-09-13   
        lxNdOTGa3-I           27         1480       2021-11-27   
AfD TV  wtAo1hE9pEc         1152         7615       2023-06-15   
        gbz_LecnBGs        14947       215714       2023-05-15   
        NZYZJ85lE2I         7595        64732       2022-06-07   
        XdKfTJZeuec         2283        18716       2019-03-28   
        3W20MPRKKiM         1815         9195       2023-06-15   
        c26W8Kr-3pE         1685        10877       2021-05-29   
        W-gBgUkLHAI        27685       328410       2021-12-15   
        xHA4qgJFXuw         1982        11341       2023-12-19   
        Kz0pA-d8yik         1884        12047       2023-03-1

In [16]:
top_videos.reset_index()[["channel", "n_sentences", "elite", "video_title"]].to_csv(
    src.OUT / "tables/most_antielitism_videos_per_channel.csv",
    index=False,
)